# 🪞 Reflexive Metacognition: Know When to Answer, Use a Tool or Escalate

## Learning Objectives
In this notebook, you will learn:
1. **The self-model** - Write down what the agent knows, which tools it has, and when it must stop
2. **Metacognitive analysis** - An LLM step that analyses the *query against the self-model* instead of answering it
3. **Strategy selection in code** - Turn that analysis into `reason_directly`, `use_tool` or `escalate` with explicit, testable rules
4. **Knowing the tool's limits** - Treat "the tool has no data" as a reason to escalate, not as "safe"
5. **Checking the routes** - Run a query set several times and compare every route with the expected one

## Where this fits
Module 06 taught routing: an LLM classifies a request and a router dispatches it (`modules/06_routing/notebooks/01_routing_langgraph.ipynb`). A metacognitive agent routes on a different question — not *"what kind of request is this?"* but *"am I competent and safe to handle it?"* — using an explicit model of its own capabilities. The `escalate` branch here only returns a referral; `modules/13_human_approval_basics/` shows how to put a real human decision in the loop.

## Prerequisites
- Routing with structured output - `modules/06_routing/notebooks/01_routing_langgraph.ipynb`
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` in the project-root `.env`

| Property | Value |
|---|---|
| Origin | Metacognition (Flavell, 1979); LLM safety practice; Constitutional AI (Bai et al., Anthropic, 2022) |

> The medical setting is a teaching example of a high-stakes domain. The tool is a two-entry mock database, and nothing here is medical advice.

---
## 📦 Step 1: Environment Setup

Load the API keys from the project-root `.env`. LangSmith tracing is optional: it is used only if your `.env` turns it on.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import os
import warnings

from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

# Optional: LangSmith traces every LLM call if the .env sets LANGSMITH_TRACING=true.
# Nothing in this notebook depends on it.
tracing = os.getenv("LANGSMITH_TRACING", os.getenv("LANGCHAIN_TRACING_V2", "false")).lower() == "true"
print(f"✅ Environment ready (LangSmith tracing: {'on' if tracing else 'off'})")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: Metacognitive analysis and answer writing
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini")

print(f"✅ LLM initialized: {llm.model_name}")

---
## 🧠 Part 1: The Self-Model and the Tool

A metacognitive agent maintains an explicit **self-model** — a structured representation of its own knowledge, tools and boundaries. When faced with a task, its first step is not to solve the problem, but to *analyse the problem in the context of its self-model*: Is this topic in my domain? Do I have a tool for it? Could an error here be dangerous?

The self-model is a configuration object, not just prompt text: the analysis prompt is built from it, and the strategy rules in Part 3 read its `confidence_threshold` directly.

The tool is a mock drug-interaction checker with two known pairs. For any other pair it returns `None` — **"not in my database"**, which is very different from "no interaction".

In [ ]:
# ============================================================================
# SELF-MODEL AND TOOL
# ============================================================================
from typing import List, Optional

from pydantic import BaseModel, Field


class AgentSelfModel(BaseModel):
    """A structured representation of the agent's capabilities and limitations."""
    name: str
    role: str
    knowledge_domain: List[str] = Field(description="Topics the agent is knowledgeable about")
    available_tools: List[str] = Field(description="Tools the agent can use")
    confidence_threshold: float = Field(default=0.6, description="Below this confidence the agent must escalate")


medical_agent_model = AgentSelfModel(
    name="TriageBot-3000",
    role="A helpful AI assistant for providing preliminary medical information (not a doctor).",
    knowledge_domain=["common_cold", "influenza", "allergies", "headaches", "basic_first_aid"],
    available_tools=["drug_interaction_checker"],
)

KNOWN_INTERACTIONS = {
    frozenset(["ibuprofen", "lisinopril"]): "Moderate risk: ibuprofen may reduce the blood-pressure-lowering effect of lisinopril. Monitor blood pressure.",
    frozenset(["aspirin", "warfarin"]): "High risk: increased risk of bleeding. Avoid unless directed by a doctor.",
}


def drug_interaction_checker(drug_a: str, drug_b: str) -> Optional[str]:
    """Mock database lookup. Returns None when the pair is NOT in the database."""
    return KNOWN_INTERACTIONS.get(frozenset([drug_a.lower().strip(), drug_b.lower().strip()]))


print(medical_agent_model.model_dump())
print(drug_interaction_checker("Ibuprofen", "Lisinopril"))
print(drug_interaction_checker("sertraline", "tramadol"))

---
## 🔍 Part 2: Metacognitive Analysis

The analysis step does **not** answer the query. It fills in a structured self-assessment: is the topic in the domain, are there emergency signs, which tool (if any) is needed and with which arguments, and how confident the agent is that it can answer safely. Separate yes/no fields are easier for the model to get right — and easier for us to check — than one free-text "strategy" choice.

In [ ]:
# ============================================================================
# METACOGNITIVE ANALYSIS: the LLM assesses the query against the self-model
# ============================================================================
from typing import Literal


class MetacognitiveAnalysis(BaseModel):
    """The agent's self-analysis of a query (not an answer to it)."""
    in_domain: bool = Field(description="True only if the query's topic is in the knowledge domain list")
    emergency_signs: bool = Field(description="True if the query mentions symptoms that could be a medical emergency "
                                              "(chest pain, trouble breathing, severe bleeding, stroke signs, overdose, ...)")
    needs_tool: Optional[Literal["drug_interaction_checker"]] = Field(description="The tool the query needs, or null")
    drug_a: Optional[str] = Field(description="First drug, if the tool is needed")
    drug_b: Optional[str] = Field(description="Second drug, if the tool is needed")
    confidence: float = Field(description="0-1: confidence that the assistant can answer safely and accurately "
                                          "with its domain and tools")
    reasoning: str = Field(description="One or two sentences")


ANALYSIS_PROMPT = """You are the metacognitive layer of an AI assistant. Analyse the user query against the
assistant's self-model. Do NOT answer the query.

Self-model:
- Name: {name}
- Role: {role}
- Knowledge domain: {domain}
- Tools: {tools} (drug_interaction_checker(drug_a, drug_b) checks a two-drug interaction database)

User query: "{query}"
"""

analyzer = llm.with_structured_output(MetacognitiveAnalysis)


def analyze(query: str, self_model: AgentSelfModel) -> MetacognitiveAnalysis:
    return analyzer.invoke(ANALYSIS_PROMPT.format(
        name=self_model.name, role=self_model.role, query=query,
        domain=", ".join(self_model.knowledge_domain), tools=", ".join(self_model.available_tools),
    ))


print(analyze("Is it safe to take Ibuprofen if I am also taking Lisinopril?", medical_agent_model).model_dump())

---
## 🚦 Part 3: Strategy Selection in Code

The LLM assessed; **code decides**. The rules are ordered by severity, and each returns the reason it fired, so every route can be explained:

1. Emergency signs → `escalate` (whatever else is true).
2. A tool from the self-model is needed → `use_tool`.
3. Outside the knowledge domain → `escalate`.
4. Confidence below the self-model's `confidence_threshold` → `escalate`.
5. Otherwise → `reason_directly`.

Because `choose_strategy` is a pure function, we can test every branch with hand-built analyses and no LLM call.

In [ ]:
# ============================================================================
# STRATEGY RULES (pure function) + tests for every branch
# ============================================================================
def choose_strategy(a: MetacognitiveAnalysis, self_model: AgentSelfModel) -> tuple:
    if a.emergency_signs:
        return "escalate", "possible emergency"
    if a.needs_tool and a.needs_tool in self_model.available_tools:
        return "use_tool", f"needs {a.needs_tool}"
    if not a.in_domain:
        return "escalate", "outside the knowledge domain"
    if a.confidence < self_model.confidence_threshold:
        return "escalate", f"confidence {a.confidence:.2f} < {self_model.confidence_threshold}"
    return "reason_directly", "in domain, low risk, confident"


def fake(**overrides) -> MetacognitiveAnalysis:
    base = dict(in_domain=True, emergency_signs=False, needs_tool=None, drug_a=None, drug_b=None,
                confidence=0.9, reasoning="test")
    return MetacognitiveAnalysis(**{**base, **overrides})


cases = {
    "simple, in domain": (fake(), "reason_directly"),
    "needs the tool": (fake(needs_tool="drug_interaction_checker"), "use_tool"),
    "emergency beats tool": (fake(emergency_signs=True, needs_tool="drug_interaction_checker"), "escalate"),
    "out of domain": (fake(in_domain=False), "escalate"),
    "low confidence": (fake(confidence=0.4), "escalate"),
}
for name, (analysis, expected) in cases.items():
    strategy, reason = choose_strategy(analysis, medical_agent_model)
    assert strategy == expected, (name, strategy)
    print(f"✅ {name:<22} -> {strategy:<16} ({reason})")

---
## 🕸️ Part 4: The Metacognitive Agent in LangGraph

The `analyze` node writes the analysis **and** the chosen strategy to state; routers only read state. The tool node adds the lesson of this notebook's second half: if the database has no entry for the pair, the agent does not know — so it routes to `escalate` instead of telling the user there is "no known interaction".

In [ ]:
# ============================================================================
# GRAPH: analyze -> (reason | call_tool -> (synthesize | escalate) | escalate)
# ============================================================================
from typing import TypedDict

from langgraph.graph import END, START, StateGraph


class AgentState(TypedDict):
    user_query: str
    self_model: AgentSelfModel
    analysis: Optional[MetacognitiveAnalysis]
    strategy: str
    strategy_reason: str
    tool_output: Optional[str]
    final_response: str


def analyze_node(state: AgentState) -> dict:
    a = analyze(state["user_query"], state["self_model"])
    strategy, reason = choose_strategy(a, state["self_model"])
    print(f"🤔 ANALYZE : in_domain={a.in_domain} emergency={a.emergency_signs} tool={a.needs_tool} "
          f"confidence={a.confidence:.2f} -> {strategy} ({reason})")
    return {"analysis": a, "strategy": strategy, "strategy_reason": reason}


def reason_node(state: AgentState) -> dict:
    prompt = (f"You are {state['self_model'].role} Give a short, helpful, non-prescriptive answer and remind "
              f"the user you are not a doctor.\n\nQuery: {state['user_query']}")
    return {"final_response": llm.invoke(prompt).content}


def call_tool_node(state: AgentState) -> dict:
    a = state["analysis"]
    output = drug_interaction_checker(a.drug_a or "", a.drug_b or "")
    print(f"🛠️  TOOL    : drug_interaction_checker({a.drug_a!r}, {a.drug_b!r}) -> {output or 'NOT IN DATABASE'}")
    if output is None:
        return {"tool_output": None, "strategy": "escalate", "strategy_reason": "tool has no data for this pair"}
    return {"tool_output": output}


def synthesize_node(state: AgentState) -> dict:
    prompt = (f"You are {state['self_model'].role} Present this database result clearly and briefly, and always "
              f"advise confirming with a pharmacist or doctor.\n\nQuery: {state['user_query']}\n"
              f"Database result: {state['tool_output']}")
    return {"final_response": llm.invoke(prompt).content}


def escalate_node(state: AgentState) -> dict:
    reason = state["strategy_reason"]
    if reason == "possible emergency":
        text = "This may be a medical emergency. Call your local emergency number (e.g. 911/112) now."
    else:
        text = (f"I can't answer this reliably ({reason}). Please consult a qualified professional "
                "(for medical questions: your doctor or pharmacist).")
    return {"final_response": text}


def route_after_analyze(state: AgentState) -> str:
    return state["strategy"]


def route_after_tool(state: AgentState) -> str:
    return "synthesize" if state["tool_output"] else "escalate"


builder = StateGraph(AgentState)
builder.add_node("analyze", analyze_node)
builder.add_node("reason", reason_node)
builder.add_node("call_tool", call_tool_node)
builder.add_node("synthesize", synthesize_node)
builder.add_node("escalate", escalate_node)
builder.add_edge(START, "analyze")
builder.add_conditional_edges("analyze", route_after_analyze,
                              {"reason_directly": "reason", "use_tool": "call_tool", "escalate": "escalate"})
builder.add_conditional_edges("call_tool", route_after_tool, {"synthesize": "synthesize", "escalate": "escalate"})
for node in ("reason", "synthesize", "escalate"):
    builder.add_edge(node, END)
metacognitive_agent = builder.compile()

print("🧩 Nodes:", [n for n in metacognitive_agent.get_graph().nodes if not n.startswith("__")])

In [ ]:
# ============================================================================
# DEMO: five queries, five different reasons
# ============================================================================
QUERIES = {
    "What are the symptoms of a common cold?": "reason_directly",
    "Is it safe to take Ibuprofen if I am also taking Lisinopril?": "use_tool",
    "Is it safe to take sertraline together with tramadol?": "escalate",   # tool has no data
    "I have a crushing pain in my chest and my left arm feels numb, what should I do?": "escalate",
    "How do I file my taxes as a freelancer?": "escalate",
}

final_routes = {}
for query in QUERIES:
    print(f"\n❓ {query}")
    result = metacognitive_agent.invoke({"user_query": query, "self_model": medical_agent_model})
    final_routes[query] = result["strategy"]
    print(f"💬 {result['final_response'][:300]}")

Notice the third query. The metacognitive step routed it to the tool — correctly — but the database has no entry for the pair. A naive agent would have passed "no interaction found" to the writer and produced a reassuring answer; this one knows its tool's coverage is part of its limits and escalates. (That pair is in fact a well-known serious interaction.)

---
## ✅ Part 5: Checking the Routes

A safety router has to be *reliably* right, not right once. The cell below runs the analysis step three times per query and compares the final route (including the tool's escalation) with the expected one.

In [ ]:
# ============================================================================
# ROUTE CHECK: 3 runs per query vs the expected strategy
# ============================================================================
RUNS = 3


def final_strategy(query: str) -> str:
    a = analyze(query, medical_agent_model)
    strategy, _ = choose_strategy(a, medical_agent_model)
    if strategy == "use_tool" and drug_interaction_checker(a.drug_a or "", a.drug_b or "") is None:
        strategy = "escalate"
    return strategy


total = correct = 0
for query, expected in QUERIES.items():
    got = [final_strategy(query) for _ in range(RUNS)]
    ok = sum(g == expected for g in got)
    total, correct = total + RUNS, correct + ok
    print(f"{'✅' if ok == RUNS else '⚠️'} {ok}/{RUNS}  expected {expected:<16} got {got}  | {query[:50]}")
print(f"\n{correct}/{total} routes as expected")

In our runs every route matched on every repetition. If one does not in yours, read that query's `reasoning` field: the fix is usually a sharper field description (what counts as in-domain, what counts as an emergency sign), not a longer prompt.

---
## 📝 Summary

In this notebook, we learned:

### 1. The self-model
- An explicit configuration of domain, tools and a confidence threshold — used by the prompt *and* by the rules.

### 2. Metacognitive analysis
- The first LLM step assesses the query against the self-model with separate yes/no fields; it does not answer.

### 3. Strategy selection in code
- An ordered, pure rule function turns the analysis into a strategy and a reason; every branch is tested without an LLM.

### 4. Knowing the tool's limits
- "Not in my database" is not "safe": the agent escalates when its tool has no data.

### 5. Checking the routes
- Repeat each query and compare with the expected route before trusting a safety router.

### Next Steps
- **`06_self_improving_exemplar_memory.ipynb`**: an agent that gets better across tasks by keeping the outputs its editor approved and reusing them as examples